# Modelling clothing shop reviews

In this notebook first we will split the data, then create pipelines for num, cat and text data. After that we will merge this pipelines and use ML models for predictions. At the end we will also fine tune our model.

## Libraries import and data preparation 

In [46]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import spacy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import FeatureUnion
from tqdm import tqdm
from sklearn.compose import ColumnTransformer

nlp = spacy.load('en_core_web_sm')

In [6]:
data = pd.read_csv("reviews.csv")
num_cols = ["Age", "Positive Feedback Count", "Recommended IND"]
cat_cols = ["Division Name", "Department Name", "Class Name"]
text_cols = ["Title", "Review Text"]
data = data.drop(["Clothing ID"], axis=1)

X = data.drop('Recommended IND', axis=1)
y = data['Recommended IND'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, shuffle=True, random_state=42)

## Preparing Pipelines


### Numerical Features Pipeline

In [41]:
num_pipeline = Pipeline([
    (
        'scaler',
        StandardScaler(),
    ),
])

num_pipeline

Pipeline(steps=[('scaler', StandardScaler())])

### Categorical Features Pipeline

In [40]:
cat_pipeline = Pipeline([
    (
        'ordinal_encoder',
        OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,
        )
    ),    
    (
        'cat_encoder',
        OneHotEncoder(
            sparse_output=False,
            handle_unknown='ignore',
        )
    ),
])
cat_pipeline

Pipeline(steps=[('ordinal_encoder',
                 OrdinalEncoder(handle_unknown='use_encoded_value',
                                unknown_value=-1)),
                ('cat_encoder',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

### Text Feature Pipeline 

#### Word Count Transformer
In the text pipeline we will create two custom features. First one will count good and bad words already chosen like good, perfect or bad.

In [7]:
class WordCount(BaseEstimator, TransformerMixin):
    def __init__(self, words: np.array):
        self.words = words
        return

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        count = 0
        for text in X:
            for word in text.split(" "):
                if word in self.words:
                    count+=1

        return count



In [39]:
positive_words = np.array(['good','perfect','awesome', 'love'])
negative_words = np.array(['hate', 'bad','badly','disappointed'])

initial_text_preprocess = Pipeline([
    (
        'dimension_reshaper',
        FunctionTransformer(
            np.reshape,
            kw_args={'newshape':-1},
        ),
    ),
])

feature_engineering = FeatureUnion([
    ('positive_words_count', WordCount(words = positive_words)),
    ('negative_words_count', WordCount(words = negative_words)),
])

character_counts_pipeline = Pipeline([
    (
        'initial_text_preprocess',
        initial_text_preprocess,
    ),
    (
        'feature_engineering',
        feature_engineering,
    ),
])

character_counts_pipeline

Pipeline(steps=[('initial_text_preprocess',
                 Pipeline(steps=[('dimension_reshaper',
                                  FunctionTransformer(func=<function reshape at 0x0000026B71383E70>,
                                                      kw_args={'newshape': -1}))])),
                ('feature_engineering',
                 FeatureUnion(transformer_list=[('positive_words_count',
                                                 WordCount(words=array(['good', 'perfect', 'awesome', 'love'], dtype='<U7'))),
                                                ('negative_words_count',
                                                 WordCount(words=array(['hate', 'bad', 'badly', 'disappointed'], dtype='<U12')))]))])

#### Sentiment Analysis Transformer

Now when we have one pipeline for word counts, let's create the second one that will analyze the sentiment and help getting better context of reviews. We have bigger data set, so we will optimize a little bit process of transforming. First, instead of bert-base model we will use tiny bert, which has lesser parameters, but for our task will do fine. Second, we will transform our data in batches of size 32, not to create massive matrix. Last, but not least we will use cuda for GPU computing, which will be much faster solution. To do it, I will use Google Colab environment for compiling our code, where there is GPU T4 available.

In [43]:
class Sentiment(BaseEstimator, TransformerMixin):
    def __init__(self, model_name='huawei-noah/TinyBERT_General_4L_312D', batch_size=32):
        self.model_name = model_name
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

    def transform(self, X):
        self.model.eval()
        all_embeddings = []
        
        # batch processing
        for i in tqdm(range(0, len(X), self.batch_size)):
            batch_texts = X[i : i + self.batch_size]
            batch_texts = batch_texts.tolist() if hasattr(batch_texts, 'tolist') else batch_texts
            
            inputs = self.tokenizer(batch_texts, return_tensors='pt', 
                                    padding=True, truncation=True, max_length=128).to(self.device)
            
            with torch.no_grad():
                outputs = self.model(**inputs)
                embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                all_embeddings.append(embeddings)
        
        return np.vstack(all_embeddings)

In [45]:
sentiment_pipeline = Pipeline([
    (
        'dimension_reshaper',
        FunctionTransformer(
            np.reshape,
            kw_args={'newshape':-1},
        ),
    ),
    (
        'lemmatizer',
        Sentiment(),
    ),
])
sentiment_pipeline 

c:\Users\Maksym\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Maksym\.cache\huggingface\hub\models--huawei-noah--TinyBERT_General_4L_312D. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


ImportError: 
AutoModel requires the PyTorch library but it was not found in your environment.
However, we were able to find a TensorFlow installation. TensorFlow classes begin
with "TF", but are otherwise identically named to our PyTorch classes. This
means that the TF equivalent of the class you tried to import would be "TFAutoModel".
If you want to use TensorFlow, please use TF classes instead!

If you really do want to use PyTorch please go to
https://pytorch.org/get-started/locally/ and follow the instructions that
match your environment.


## Combine Pipelines

Now, we will combine all the pipelines we created above.

In [47]:
feature_engineering = ColumnTransformer([
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols),
        ('character_counts', character_counts_pipeline, text_cols),
        ('sentiment_analysis', sentiment_pipeline, ["Review Text"]),
])

feature_engineering

NameError: name 'sentiment_pipeline' is not defined